# SOXX 官方流通份额：原始 SEC 拆分对照

仅用已备份的 2011 年原始 SEC 文件及当前 iShares 工作簿，对照同一报告日份额。冻结计划见 `plan.json`；零市场结果、零拟合、零账户。

In [1]:
import sys,json,hashlib,re,datetime,math
from pathlib import Path
from bs4 import BeautifulSoup
from xml.etree import ElementTree as ET
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
card=root/'research/stage1_soxx_share_original_split_control_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen']
sha=lambda b:hashlib.sha256(b).hexdigest()
assert not (card/'run_state.json').exists()
prior=json.loads((root/src['SEC_prior_source_result']).read_text());event=[r for r in prior['reports'] if r['accession']==src['SEC_accession']];assert len(event)==1 and event[0]['raw_sha256']==src['SEC_sha256'] and event[0]['accepted_at_utc']==src['SEC_accepted_at_utc'] and event[0]['source_NAS_scoped_restore_verified'] and event[0]['source_NAS_archive_sha256']==src['SEC_NAS_archive_sha256']
receipt=json.loads((root/src['issuer_workbook_NAS_receipt']).read_text());assert receipt['verified'] and receipt['sha256']==src['issuer_workbook_NAS_sha256'] and receipt['members'][src['issuer_workbook']]==src['issuer_workbook_sha256']
state={'plan_sha256':sha((card/'plan.json').read_bytes()),'phase':'source_read_attempted','SEC_original_raw_read_attempts':1,'issuer_workbook_raw_read_attempts':1,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
sec=(root/src['SEC_original']).read_bytes();workbook=(root/src['issuer_workbook']).read_bytes();assert sha(sec)==src['SEC_sha256'] and sha(workbook)==src['issuer_workbook_sha256']
soup=BeautifulSoup(sec.decode('utf-8',errors='replace'),'html.parser');matches=[]
for table in soup.find_all('table'):
 text=' '.join(table.stripped_strings)
 if 'Shares outstanding' not in text or 'SOX Semiconductor' not in text:continue
 headings=[];shares=[]
 for row in table.find_all('tr'):
  cells=[c.get_text(' ',strip=True) for c in row.find_all(['td','th'],recursive=False)];nonempty=[v for v in cells if v]
  line=' '.join(nonempty)
  if 'SOX Semiconductor' in line and 'Natural' in line and 'NYSE' in line:headings=nonempty
  if re.match(r'^Shares outstanding',line,re.I):shares=nonempty
 if headings and shares:matches.append((headings,shares))
assert len(matches)==1,(len(matches),matches[:2]);headings,shares=matches[0]
assert len(headings)==3 and len(shares)==4 and 'SOX Semiconductor' in headings[1]
original_count=int(shares[2].replace(',',''));assert original_count>0
xmltext=workbook.decode('utf-8');repaired,count=re.subn(r'&(?!amp;|lt;|gt;|quot;|apos;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;',xmltext);assert count==6
xml=ET.fromstring(repaired);ns={'s':'urn:schemas-microsoft-com:office:spreadsheet'};key='{urn:schemas-microsoft-com:office:spreadsheet}Name'
sheets={w.attrib[key]:w.findall('.//s:Row',ns) for w in xml.findall('s:Worksheet',ns)};rows=sheets['Historical'];rowvals=lambda row:[c.findtext('s:Data',default='',namespaces=ns) for c in row.findall('s:Cell',ns)]
assert rowvals(rows[0])==['As Of','NAV per Share','Ex-Dividends','Shares Outstanding']
matched=[rowvals(row) for row in rows[1:] if rowvals(row)[0]=='Jan 31, 2011'];assert len(matched)==1 and len(matched[0])==4
current_count=int(matched[0][3].replace(',',''));ratio=current_count/original_count
result={'decision':'original_SEC_2011_share_count_vs_current_workbook_split_control','plan_sha256':sha((card/'plan.json').read_bytes()),'SEC_accession':src['SEC_accession'],'SEC_accepted_at_utc':src['SEC_accepted_at_utc'],'SEC_report_date':src['SEC_report_date'],'SEC_fund_header':headings[1],'SEC_original_shares_outstanding':original_count,'issuer_current_workbook_shares_on_same_date':current_count,'current_to_original_ratio':ratio,'announced_2024_split_ratio':3.0,'ratio_matches_announced_split':bool(math.isclose(ratio,3.0,rel_tol=0,abs_tol=1e-12)),'source_NAS_prior_verified':True,'first_public_workbook_clock_qualified':False,'full_history_adjustment_qualified':False,'creation_flow_feature_qualified':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n');summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['One original quarterly report date and one current workbook value cannot prove full-history revisions or daily first-public times.','No Stage1 feature or account was run.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(result)

{'decision': 'original_SEC_2011_share_count_vs_current_workbook_split_control', 'plan_sha256': '47678ce8bf88488cce60321f55c8a6b601135be0fff9a1701ca8df8ea84e7b3e', 'SEC_accession': '0001193125-11-093502', 'SEC_accepted_at_utc': '2011-04-11T20:08:56.000Z', 'SEC_report_date': '2011-01-31', 'SEC_fund_header': 'iShares PHLX\xa0SOX Semiconductor Sector\xa0Index\xa0Fund', 'SEC_original_shares_outstanding': 3750000, 'issuer_current_workbook_shares_on_same_date': 11250000, 'current_to_original_ratio': 3.0, 'announced_2024_split_ratio': 3.0, 'ratio_matches_announced_split': True, 'source_NAS_prior_verified': True, 'first_public_workbook_clock_qualified': False, 'full_history_adjustment_qualified': False, 'creation_flow_feature_qualified': False, 'SOXX_market_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0}


# SOXX 原始份额与当前工作簿：拆分对照

已备份的原始 SEC N-CSRS（accession `0001193125-11-093502`，2011-04-11 20:08:56 UTC 接收）在 2011-01-31 的 SOXX 专属报表列记载 **3,750,000 股**；当前 iShares 历史工作簿同日报 **11,250,000 股**，严格等于 **3 倍**。[iShares 官方招募说明书](https://www.ishares.com/us/literature/prospectus/p-ishares-phlx-semiconductor-etf-3-31.pdf)记载 SOXX 于 2024-03-06 收盘后 3 比 1 拆分。原件身份、基金列选择、份额行及两个先前 NAS 原件归档回执均经校验；零市场结果、拟合及账户。

**结论：当前下载至少在这个 2011 原始对照点使用了未来拆分后的份额数量。** 该单点不能证明全历史调整规则，也不能证明每日份额首次发布时间。尤其不能把 2011 年的 1,125 万股当作当时发行人原始报告的绝对份额，或据此直接训练申赎特征。份额比率可能对统一拆分因子不敏感，但历史修订与可用时刻仍未认证。

下一步只有在取得历史原始每日发布版本或明确可审计的披露时钟、并核对分拆前后连续性后，才另行冻结已发布日份额变化的 Stage1 增量信息试验；通过后仍需完整 0bp Tiered 费用账户、2/5bp 与独立前向证据。

这个原始 SEC 单点增加了拆分倍数的独立文档对照；[此前更完整的发行人版本审计](../soxx_issuer_shares_source_20261022/conclusion.md)已有三个历史网页同样的 3 倍对照，并且[固定份额信息门](../stage1_issuer_share_creation_information_20261022/conclusion.md)已经失败。本卡只补来源证据，不重开该机制。
